# Ran on kaggle due to lack of ram on my laptop

In [ ]:
!pip install -q -U peft sentence-transformers bitsandbytes accelerate

In [ ]:
import os
import json
import torch
import gc
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig
from sentence_transformers import CrossEncoder
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

# --- 1. AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# --- 2. KAGGLE PATHS ---
RAW_RESULTS_PATH = "/kaggle/input/datasets/vasanthsubramanian01/fyp-vasanth/raw_retrieval_results.json" 
CV_DATA_PATH = "/kaggle/input/datasets/vasanthsubramanian01/fyp-vasanth/processed_eval_data.json"

# NEW: Update this if you upload your previous 350-CV CSV as a dataset
PREVIOUS_RESULTS_PATH = "/kaggle/input/datasets/vasanthsubramanian01/output-1/retrieval_metrics_final_ensemble.csv"
OUTPUT_CSV = "/kaggle/working/retrieval_metrics_balanced.csv"

# --- 3. SETTINGS ---
CAT_LIMIT = 25       # 25 per category
TOP_K_RESULTS = 10 
SAVE_EVERY = 25      # More frequent saves for safety

# --- 4. LOAD MODELS ---
gc.collect()
torch.cuda.empty_cache()

print("⏳ Loading BGE-Reranker on CPU...")
bge_model = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512, device='cpu')

print("⏳ Spreading Llama-3.1-Storm across Dual GPUs...")
base_id = "akjindal53244/Llama-3.1-Storm-8B"
adapter_id = "LlamaFactoryAI/cv-job-description-matching"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(base_id)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_id, 
    quantization_config=bnb_config, 
    device_map="auto",             
    low_cpu_mem_usage=True
)

# Patching adapter configuration
print("🔧 Manually patching adapter configuration...")
config_file = hf_hub_download(repo_id=adapter_id, filename="adapter_config.json")
with open(config_file, 'r') as f:
    config_dict = json.load(f)
config_dict['task_type'] = "CAUSAL_LM"
adapter_config = LoraConfig(**config_dict)

llama_model = PeftModel.from_pretrained(base_model, adapter_id, config=adapter_config)
llama_model.eval()

def get_llama_score(cv_text, jd_text):
    system_prompt = (
        "You are an advanced AI model designed to analyze CV/JD compatibility. "
        "Output ONLY JSON with matching_analysis, description, score (0-100), and recommendation."
    )
    user_content = f"<CV> {cv_text[:1000]} </CV>\n<job_description> {jd_text[:1000]} </job_description>"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(base_model.device)
    
    with torch.no_grad():
        outputs = llama_model.generate(
            **inputs, 
            max_new_tokens=128, 
            do_sample=False,
            use_cache=True, # Speed optimization
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    try:
        score_match = re.search(r'"score":\s*(\d+)', response)
        return int(score_match.group(1)) / 100.0 if score_match else 0.5
    except:
        return 0.5

# --- 5. EVALUATION ENGINE ---
def run_evaluation():
    # 1. SETUP RESUME LOGIC
    rows = []
    processed_filenames = set()

    # Load from the new output if it exists (for restarts)
    if os.path.exists(OUTPUT_CSV):
        existing_df = pd.read_csv(OUTPUT_CSV)
        rows = existing_df.to_dict('records')
        processed_filenames = set(existing_df['filename'].unique())
        print(f"🔄 Found {len(processed_filenames)} in current session output.")

    # Load from previous 12-hour run if provided
    if PREVIOUS_RESULTS_PATH and os.path.exists(PREVIOUS_RESULTS_PATH):
        prev_df = pd.read_csv(PREVIOUS_RESULTS_PATH)
        processed_filenames.update(set(prev_df['filename'].unique()))
        # If rows is empty (fresh run), start with previous data
        if not rows:
            rows = prev_df.to_dict('records')
        print(f"🔄 Integrated {len(processed_filenames)} total from previous runs.")

    # 2. LOAD & BALANCE DATA
    with open(RAW_RESULTS_PATH, "r") as f:
        full_data = json.load(f)
    
    # Filter for 25 per category
    df_raw = pd.DataFrame(full_data)
    balanced_df = df_raw.groupby('ground_truth', group_keys=False).apply(lambda x: x.head(CAT_LIMIT))
    data = balanced_df.to_dict('records')
    
    print(f"📊 Balanced Data: {len(data)} total samples ({CAT_LIMIT} per category).")

    with open(CV_DATA_PATH, "r") as f:
        cv_texts = {item['filename']: item['resume_text'] for item in json.load(f)}

    # 3. PROCESSING LOOP
    for idx, entry in enumerate(tqdm(data, desc="📊 Balanced Ensemble Evaluation")):
        filename = entry["filename"]
        
        if filename in processed_filenames:
            continue
            
        gt_category = entry["ground_truth"].upper()
        resume = cv_texts.get(filename, "")
        
        for run in entry["eval_runs"]:
            config_label = f"NER:{run['config']['ner']}_LLM:{run['config']['llm']}"
            matches = run.get("results", [])[:TOP_K_RESULTS]
            num_found = len(matches)

            m = {
                "filename": filename, "category": gt_category, "config": config_label,
                "search_success": 1 if num_found > 0 else 0, "count_retrieved": num_found,
                "hit_at_1": 0, "hit_at_5": 0, "hit_at_10": 0, "mrr": 0, "in_cat_ratio": 0,
                "avg_bge_score": 0, "llama_top_score": 0, "consensus_score": 0
            }

            if num_found > 0:
                rel_count = 0
                for i, job in enumerate(matches):
                    is_rel = (gt_category in job['title'].upper() or gt_category in job['description'].upper())
                    if is_rel:
                        rel_count += 1
                        if i == 0: m["hit_at_1"] = 1
                        if i < 5: m["hit_at_5"] = 1
                        if i < 10: m["hit_at_10"] = 1
                        if m["mrr"] == 0: m["mrr"] = round(1 / (i + 1), 4)
                
                m["in_cat_ratio"] = round(rel_count / num_found, 4)
                
                # BGE Scoring (CPU)
                pairs = [[resume[:1000], j['description'][:1000]] for j in matches]
                bge_raw = bge_model.predict(pairs)
                bge_norm = 1 / (1 + np.exp(-np.array(bge_raw)))
                m["avg_bge_score"] = round(float(np.mean(bge_norm)), 4)

                # Llama Specialist (GPU) - Scoring Top Result
                m["llama_top_score"] = get_llama_score(resume, matches[0]['description'])
                m["consensus_score"] = round((m["avg_bge_score"] + m["llama_top_score"]) / 2, 4)

            rows.append(m)

        if (idx + 1) % SAVE_EVERY == 0:
            pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)

    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
    print("✅ Evaluation Cycle Complete!")

if __name__ == "__main__":
    run_evaluation()